In [ ]:
import os

from data_index.iceberg_config import S3TablesCatalogConfig, IcebergTableConfig

os.environ["AWS_PROFILE"] = "edge-admin"

In [ ]:
# --- Sink config ---
data_index_catalog_config = S3TablesCatalogConfig(
    region="ap-southeast-2",
    arn="arn:aws:s3tables:ap-southeast-2:704910415367:bucket/data-index",
)

structured_metadata_table_config = IcebergTableConfig(
    catalog_config=data_index_catalog_config,
    namespace="data_index",
    table_name=f"structured_metadata_v1",
)

unstructured_metadata_table_config = IcebergTableConfig(
    catalog_config=data_index_catalog_config,
    namespace="data_index",
    table_name="unstructured_metadata",
)

In [ ]:
structured_metadata = structured_metadata_table_config.load()

In [ ]:
structured_metadata.metadata

In [ ]:
structured_metadata.schema().model_dump()

# Full structured_metadata table

In [ ]:
con = structured_metadata.scan().to_duckdb(table_name="structured_metadata")

In [ ]:
con.execute("""
    SELECT COUNT(*)
    FROM structured_metadata
"""
).fetchone()

In [ ]:
con.execute("""
    SELECT *
    FROM structured_metadata
    ORDER BY s3_uri
"""
).fetchdf()

In [ ]:
con.execute("""
    SELECT regexp_extract(s3_uri, 's3://imos-data/([^/]+/[^/]+)', 1) as prefix,
           COUNT(*) as n_files,
           COUNT(DISTINCT site_code) as n_sites,
           COUNT(DISTINCT platform_code) as n_platforms,
           COUNT(DISTINCT deployment_code) as n_deployments,
           COUNT(DISTINCT instrument) as n_instruments
    FROM structured_metadata
    GROUP BY prefix
    ORDER BY prefix
"""
).fetchdf()

# Moorings queries

In [ ]:
con.execute("""
    CREATE OR REPLACE VIEW anmn_all AS
    SELECT *,
           regexp_extract(s3_uri, 's3://imos-data/(.*)', 1) as s3_key,
           regexp_extract(s3_uri, 's3://imos-data/IMOS/ANMN/([^/]+)', 1) as subfac,
           regexp_extract(s3_uri, 'Temperature|(CTD|Biogeochem)_(timeseries|profiles)|Velocity|Wave|CO2|Meteorology|aggregated_products|(aggregated|hourly|gridded)_timeseries') as data_category,
           regexp_extract(file_version, 'Level ([0-4])', 1)::INT1 as fv,
           regexp_matches(s3_uri, 'real[-_]?time', 'i') as is_realtime
    FROM structured_metadata
    WHERE s3_uri LIKE 's3://imos-data/IMOS/ANMN/%'
"""
)
con.execute("""
    SELECT *
    FROM anmn_all
"""
).fetchdf()

In [ ]:
# Sub-facility summary
con.execute("""
    SELECT subfac,
           COUNT(*) as n_files,
           COUNT(DISTINCT site_code) as n_sites,
           COUNTIF(fv = 0)::INT as n_fv0,
           COUNTIF(fv = 1)::INT as n_fv1,
           COUNTIF(fv = 2)::INT as n_fv2,
           COUNTIF(is_realtime)::INT as n_realtime
    FROM anmn_all
    GROUP BY subfac
    ORDER BY subfac
"""
).fetchdf()

In [ ]:
# By data category
con.execute("""
    SELECT subfac,
           data_category,
           COUNT(*) as n_files,
           COUNT(DISTINCT site_code) as n_sites,
           COUNTIF(fv = 0)::INT as n_fv0,
           COUNTIF(fv = 1)::INT as n_fv1,
           COUNTIF(fv = 2)::INT as n_fv2
    FROM anmn_all
    GROUP BY subfac, data_category
    ORDER BY subfac, data_category
"""
).fetchdf()

## Moorings products

In [ ]:
# Create view of all hourly products
con.execute("""
    CREATE OR REPLACE VIEW anmn_hourly_timeseries AS
    SELECT
        subfac,
        site_code,
        time_min,
        time_max,
        variables,
        s3_key
    FROM anmn_all
    WHERE data_category = 'hourly_timeseries'
""")
con.execute("SELECT * FROM anmn_hourly_timeseries ORDER BY s3_key").fetchdf()

In [ ]:
# Create view of latest data for each site & variable
    # CREATE OR REPLACE VIEW source_files AS
con.execute("""
WITH vars AS (
    SELECT
        s3_key,
        site_code,
        unnest(variables) AS var,
        time_min,
        time_max
    FROM anmn_all
    WHERE fv = 1 AND
      NOT is_realtime AND
      data_category NOT IN ('CTD_profiles', 'Biogeochem_profiles', 'aggregated_timeseries', 'CO2')
), vars_grouped AS (
    SELECT
        site_code,
        var,
        count(DISTINCT s3_key) as n_files,
        min(time_min) as time_min,
        max(time_max) as time_max
    FROM vars
    WHERE var IN ('TEMP', 'PSAL', 'UCUR')
    GROUP BY site_code, var
    ORDER BY site_code, var
)
SELECT
    v.site_code,
    list(v.var) AS vars,
    max(date(v.time_max)) as source_data_end,
    date(h.time_max) as product_data_end,
    h.s3_key as product_s3_key
FROM vars_grouped v LEFT JOIN anmn_hourly_timeseries h ON v.site_code = h.site_code AND v.var IN h.variables
GROUP BY v.site_code, product_data_end, product_s3_key
HAVING source_data_end > product_data_end OR product_s3_key IS NULL
ORDER BY v.site_code, product_data_end, product_s3_key
""").fetchdf()

## Find duplicates